## 模块 8：门店月度销售额累计与移动平均计算（窗口函数高级）

### 业务需求

基于原始表，统计每个门店的月度销售额，计算累计销售额与 3 个月移动平均销售额

### 思路分析

1. 直接关联销售订单表、门店信息表，按门店 + 月份分组聚合销售额

2. 用`SUM() OVER()`窗口函数计算累计销售额

3. 用`AVG() OVER()`窗口函数计算 3 个月移动平均销售额

### 核心知识点

- 窗口函数`SUM() OVER()`累计求和

- 窗口函数`AVG() OVER()`移动平均计算

- 分区窗口`PARTITION BY`使用

- 时间序列高级分析

## SQL 代码

In [1]:
import pandas as pd
import sqlite3

# 1. 连接SQLite数据库
conn = sqlite3.connect("/kaggle/input/datasets/tclaim/retail-sales/retail_sales.db")

# 4. 封装通用SQL执行函数，自动打印表格结果
def query_sql(sql):
    return pd.read_sql(sql, conn)
    
print("数据导入完成，调用 query_sql('SQL语句') 即可执行查询")

数据导入完成，调用 query_sql('SQL语句') 即可执行查询


In [2]:
sql = '''
WITH shop_monthly_sales AS (
    SELECT
        s.门店ID,
        s.门店名称,
        STRFTIME('%Y-%m', o.销售日期) AS 统计月份,
        SUM(o.销售数量 * o."单价(元)") AS 月度销售额
    FROM "销售订单表" o
    INNER JOIN "门店信息表" s ON o.门店ID = s.门店ID
    GROUP BY s.门店ID, s.门店名称, 统计月份
)
SELECT
    门店ID,
    门店名称,
    统计月份,
    月度销售额,
    SUM(月度销售额) OVER (PARTITION BY 门店ID ORDER BY 统计月份) AS 累计销售额,
    ROUND(AVG(月度销售额) OVER (PARTITION BY 门店ID ORDER BY 统计月份 ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS `3个月移动平均销售额`
FROM shop_monthly_sales
ORDER BY 门店ID, 统计月份;
'''
query_sql(sql)

,门店ID,门店名称,统计月份,月度销售额,累计销售额,3个月移动平均销售额
0,S002,贺州门店2,2025-06,261.5,261.5,261.50
1,S003,黔西南布依族苗族自治州门店3,2025-05,1950.0,1950.0,1950.00
2,S004,昆明门店4,2025-02,2489.0,2489.0,2489.00
3,S004,昆明门店4,2025-05,60.0,2549.0,1274.50
4,S005,丹东门店5,2025-01,320403.0,320403.0,320403.00
...,...,...,...,...,...,...
452,S457,济宁门店457,2025-02,129.0,129.0,129.00
453,S459,吕梁门店459,2025-01,143.0,143.0,143.00
454,S459,吕梁门店459,2025-02,810.0,953.0,476.50
455,S459,吕梁门店459,2025-03,2518.0,3471.0,1157.00


### 结果说明

输出每个门店的月度销售额、累计销售额与3个月移动平均销售额，可直接用于门店运营趋势分析、销售预测与业绩评估